In [9]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver


In [10]:
load_dotenv()

True

In [11]:
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

In [12]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [13]:
def generate_joke(state: JokeState):
    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [14]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [15]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [16]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'topic': 'pizza',
 'joke': [{'type': 'text',
   'text': 'Why does a pizza maker never get into arguments?\n\nBecause they know how to **let it slide**!',
   'extras': {'signature': 'EnMKcQFpFH0TC51GIxZrJqeK54i8mX/CA4g9MFxyharVsf3Al0vF/muydFFBoB3r0TH1etKNVPaa+tsiCk1SVCz6+A2HSEgltM5CvwrdVdtDT+lVB/VTuWNRlorktzER6iEXjrPQehl4n5qWrZ48Z4UyCax/'}}],
 'explanation': [{'type': 'text',
   'text': 'This joke relies on a **pun**, which is a form of wordplay that uses multiple meanings of a phrase to create a humorous effect.\n\nHere is the breakdown of why it is funny:\n\n1.  **The Literal Meaning (The Pizza Context):** When a pizza maker prepares a pizza, they use a tool called a **peel** (a long-handled wooden or metal paddle) to slide the pizza into and out of a hot oven. Therefore, "letting it slide" is a literal, everyday action in their profession.\n2.  **The Idiomatic Meaning:** To "let something slide" is a common English idiom that means to ignore a mistake, overlook a minor conflict, or 

In [17]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'Why does a pizza maker never get into arguments?\n\nBecause they know how to **let it slide**!', 'extras': {'signature': 'EnMKcQFpFH0TC51GIxZrJqeK54i8mX/CA4g9MFxyharVsf3Al0vF/muydFFBoB3r0TH1etKNVPaa+tsiCk1SVCz6+A2HSEgltM5CvwrdVdtDT+lVB/VTuWNRlorktzER6iEXjrPQehl4n5qWrZ48Z4UyCax/'}}], 'explanation': [{'type': 'text', 'text': 'This joke relies on a **pun**, which is a form of wordplay that uses multiple meanings of a phrase to create a humorous effect.\n\nHere is the breakdown of why it is funny:\n\n1.  **The Literal Meaning (The Pizza Context):** When a pizza maker prepares a pizza, they use a tool called a **peel** (a long-handled wooden or metal paddle) to slide the pizza into and out of a hot oven. Therefore, "letting it slide" is a literal, everyday action in their profession.\n2.  **The Idiomatic Meaning:** To "let something slide" is a common English idiom that means to ignore a mistake, overlook a minor con

In [18]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'Why does a pizza maker never get into arguments?\n\nBecause they know how to **let it slide**!', 'extras': {'signature': 'EnMKcQFpFH0TC51GIxZrJqeK54i8mX/CA4g9MFxyharVsf3Al0vF/muydFFBoB3r0TH1etKNVPaa+tsiCk1SVCz6+A2HSEgltM5CvwrdVdtDT+lVB/VTuWNRlorktzER6iEXjrPQehl4n5qWrZ48Z4UyCax/'}}], 'explanation': [{'type': 'text', 'text': 'This joke relies on a **pun**, which is a form of wordplay that uses multiple meanings of a phrase to create a humorous effect.\n\nHere is the breakdown of why it is funny:\n\n1.  **The Literal Meaning (The Pizza Context):** When a pizza maker prepares a pizza, they use a tool called a **peel** (a long-handled wooden or metal paddle) to slide the pizza into and out of a hot oven. Therefore, "letting it slide" is a literal, everyday action in their profession.\n2.  **The Idiomatic Meaning:** To "let something slide" is a common English idiom that means to ignore a mistake, overlook a minor co

# Time Travel

In [20]:
# Get state from perticular checkpoint
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1c22ae-1968-6527-8000-b0a10c56d142"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1c22ae-1968-6527-8000-b0a10c56d142'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-10-07T08:41:25.434901+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c22ae-1966-623a-bfff-4efcdcad87a7'}}, tasks=(PregelTask(id='12672fad-a223-fa5d-adca-cbb26225bc41', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': [{'type': 'text', 'text': 'Why does a pizza maker never get into arguments?\n\nBecause they know how to **let it slide**!', 'extras': {'signature': 'EnMKcQFpFH0TC51GIxZrJqeK54i8mX/CA4g9MFxyharVsf3Al0vF/muydFFBoB3r0TH1etKNVPaa+tsiCk1SVCz6+A2HSEgltM5CvwrdVdtDT+lVB/VTuWNRlorktzER6iEXjrPQehl4n5qWrZ48Z4UyCax/'}}]}),), interrupts=())

In [21]:
# Invoke graph from perticuar node
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f1c22ae-1968-6527-8000-b0a10c56d142"}})

{'topic': 'pizza',
 'joke': [{'type': 'text',
   'text': 'Why did the pizza maker go into business?\n\nBecause he wanted to make some **dough**!',
   'extras': {'signature': 'EnMKcQFpFH0TIgCl35DVMeGJQf4ngT1HcKndXZnbLz7E1Pa0pBYi2Ol6FEPYDO9fykuFN+FA1d6HWC8V3j6Hj4RS/fUK2/3iHlq0y/EfCDKyVM2hk6BMV3c6BLqEgw27AIt4c13gv8NK3oreZNB1klCOCvAD'}}],
 'explanation': [{'type': 'text',
   'text': 'This joke relies on a **pun** (a play on words) involving the double meaning of the word "dough."\n\nHere is the breakdown:\n\n1.  **Literal Meaning:** In the context of making pizza, "dough" is the essential ingredient (flour, water, and yeast) used to make the crust.\n2.  **Slang Meaning:** "Dough" is a very common informal term for **money**.\n\nThe humor comes from the fact that the pizza maker is literally working with dough to make pizza, but he is also using the word to express his motivation for starting a business: to earn money. By using the word "dough," the joke allows the answer to be true in two 

In [22]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'Why did the pizza maker go into business?\n\nBecause he wanted to make some **dough**!', 'extras': {'signature': 'EnMKcQFpFH0TIgCl35DVMeGJQf4ngT1HcKndXZnbLz7E1Pa0pBYi2Ol6FEPYDO9fykuFN+FA1d6HWC8V3j6Hj4RS/fUK2/3iHlq0y/EfCDKyVM2hk6BMV3c6BLqEgw27AIt4c13gv8NK3oreZNB1klCOCvAD'}}], 'explanation': [{'type': 'text', 'text': 'This joke relies on a **pun** (a play on words) involving the double meaning of the word "dough."\n\nHere is the breakdown:\n\n1.  **Literal Meaning:** In the context of making pizza, "dough" is the essential ingredient (flour, water, and yeast) used to make the crust.\n2.  **Slang Meaning:** "Dough" is a very common informal term for **money**.\n\nThe humor comes from the fact that the pizza maker is literally working with dough to make pizza, but he is also using the word to express his motivation for starting a business: to earn money. By using the word "dough," the joke allows the answer to be t

## Updating State

In [23]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1c22ae-1968-6527-8000-b0a10c56d142", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1c22ba-aed6-60dd-8001-01faec40a61c'}}

In [24]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c22ba-aed6-60dd-8001-01faec40a61c'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-10-07T08:47:03.226168+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c22ae-1968-6527-8000-b0a10c56d142'}}, tasks=(PregelTask(id='70ff3b15-a2f7-4e1f-6ad9-cd8b6a2e028e', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'Why did the pizza maker go into business?\n\nBecause he wanted to make some **dough**!', 'extras': {'signature': 'EnMKcQFpFH0TIgCl35DVMeGJQf4ngT1HcKndXZnbLz7E1Pa0pBYi2Ol6FEPYDO9fykuFN+FA1d6HWC8V3j6Hj4RS/fUK2/3iHlq0y/EfCDKyVM2hk6BMV3c6BLqEgw27AIt4c13gv8NK3oreZNB1klCOCvAD'}}], 'explanation': [{'type': 'te

In [25]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f1c22ba-aed6-60dd-8001-01faec40a61c"}})

{'topic': 'samosa',
 'joke': [{'type': 'text',
   'text': "Why did the samosa go to the doctor?\n\nBecause it was feeling a little **stuffed** and couldn't stop **leaking its secrets!**",
   'extras': {'signature': 'EnMKcQFpFH0TgxKYKK1cSrkjuLk0uOdFru8U1k9qp1My1pNXXqHZkE2DbzXyQ4Gh+kUrSHz8h9Iad2ABoVGmrO94fYGA67oW+/mE/AZY8X3B7iOv6tr5UEfbX8o01Ni+9Y9a3IxZakf5kcfmoOFMwrTpp+1U'}}],
 'explanation': [{'type': 'text',
   'text': 'This joke relies on a **double entendre**—using words that have both a literal and a figurative meaning—to connect the nature of a samosa with common human experiences.\n\nHere is the breakdown:\n\n**1. "Feeling a little stuffed"**\n*   **Literal meaning:** A samosa is a pastry filled with spiced potatoes, peas, and other ingredients. When it is full of filling, it is physically "stuffed."\n*   **Figurative meaning:** When a person says they are "stuffed," they usually mean they have eaten too much food and feel uncomfortably full.\n\n**2. "Leaking its secrets"**\n*   *

In [26]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': [{'type': 'text', 'text': "Why did the samosa go to the doctor?\n\nBecause it was feeling a little **stuffed** and couldn't stop **leaking its secrets!**", 'extras': {'signature': 'EnMKcQFpFH0TgxKYKK1cSrkjuLk0uOdFru8U1k9qp1My1pNXXqHZkE2DbzXyQ4Gh+kUrSHz8h9Iad2ABoVGmrO94fYGA67oW+/mE/AZY8X3B7iOv6tr5UEfbX8o01Ni+9Y9a3IxZakf5kcfmoOFMwrTpp+1U'}}], 'explanation': [{'type': 'text', 'text': 'This joke relies on a **double entendre**—using words that have both a literal and a figurative meaning—to connect the nature of a samosa with common human experiences.\n\nHere is the breakdown:\n\n**1. "Feeling a little stuffed"**\n*   **Literal meaning:** A samosa is a pastry filled with spiced potatoes, peas, and other ingredients. When it is full of filling, it is physically "stuffed."\n*   **Figurative meaning:** When a person says they are "stuffed," they usually mean they have eaten too much food and feel uncomfortably full.\n\n**2. "Leaking its secret

# Fault Tolerance

In [2]:
import time

In [3]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str
    step3: str

In [4]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(30)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"step3": "done"}

In [5]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [6]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
✅ Step 3 executed


In [7]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


🔁 Re-running the graph to demonstrate fault tolerance...

✅ Final State: {'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}


In [8]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c2284-b10d-68e4-8003-6e0ff5dfc6c9'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-10-07T08:22:53.907162+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c2284-b109-6e6d-8002-c621cf63f897'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=('step_3',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c2284-b109-6e6d-8002-c621cf63f897'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-07T08:22:53.905618+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c2283-92ec-6cbd-8001-85ee03da3fe7'}}, tasks=(PregelTask(id='072e222a-c48c-67c1